# 02 DOG2 clinical metadata and sample matching

Purpose: inspect the DOG2 clinical Excel file, identify endpoint columns, and match clinical samples to GSE238110 expression columns.

In [1]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

print("Project root:", PROJECT_ROOT)
print("Raw data dir:", RAW_DIR)
print("Processed data dir:", PROCESSED_DIR)

Project root: C:\Users\olegk\Desktop\paper4_sarcoma_dog
Raw data dir: C:\Users\olegk\Desktop\paper4_sarcoma_dog\data\raw
Processed data dir: C:\Users\olegk\Desktop\paper4_sarcoma_dog\data\processed


In [2]:
def find_file(folder, patterns):
    folder = Path(folder)
    hits = []
    for pattern in patterns:
        hits.extend(folder.glob(pattern))
    hits = sorted(set(hits))
    if not hits:
        raise FileNotFoundError(f"No matching files in {folder}. Patterns: {patterns}")
    if len(hits) > 1:
        print("Multiple candidates found:")
        for p in hits:
            print(" ", p.name)
        print("Using:", hits[0].name)
    return hits[0]

clinical_path = find_file(
    RAW_DIR / "canine_clinical_DOG2",
    ["*supplementary*table*s10*.xlsx", "*suppts10*.xlsx", "*.xlsx"]
)

print("Clinical file:", clinical_path)
xls = pd.ExcelFile(clinical_path)
print("Sheets:", xls.sheet_names)

Clinical file: C:\Users\olegk\Desktop\paper4_sarcoma_dog\data\raw\canine_clinical_DOG2\ccr-24-1854_supplementary_table_s10_suppts10.xlsx
Sheets: ['TME_subtype_SupplTable10']


In [3]:
sheets = {}
for sheet in xls.sheet_names:
    df = pd.read_excel(clinical_path, sheet_name=sheet)
    sheets[sheet] = df
    print(sheet, df.shape)
    display(df.head())

clinical_sheet = max(sheets, key=lambda s: sheets[s].shape[0] * sheets[s].shape[1])
clinical_raw = sheets[clinical_sheet].copy()

print("Selected sheet:", clinical_sheet)
print("Shape:", clinical_raw.shape)
display(pd.DataFrame({"column": clinical_raw.columns}))

TME_subtype_SupplTable10 (186, 16)


,Patient ID,Tumor Location,Site,age,weight,breed,gender,PH,ALP,Group,DFS_time,DFS_status,OS_time,OS_status,treatment,primary_immune_subtype
0,514,Left distal radius,UW,5.5,42.7,Mixed Breed,Spayed Female,0,Normal,NPHNALP,33,0,117,1,SOC,ID
1,518,Left proximal humerus,UW,8.3,44.3,Rottweiler,Spayed Female,1,Elevated,PHEALP,12,0,248,0,SOC,IE-ECM
2,619,Left distal tibia,OSU,7.5,30.8,Mixed Breed,Spayed Female,0,Normal,NPHNALP,63,1,622,0,SOC,IE-ECM
3,639,Left distal femur,OSU,6.9,36.2,Greyhound,Castrated Male,0,Normal,NPHNALP,234,1,283,1,SOC,ID
4,702,Right distal ulna,UIL,9.5,33.2,Labrador Retriever,Castrated Male,0,Normal,NPHNALP,153,1,171,0,SOC,IE-ECM


Selected sheet: TME_subtype_SupplTable10
Shape: (186, 16)


,column
0,Patient ID
1,Tumor Location
2,Site
3,age
4,weight
5,breed
6,gender
7,PH
8,ALP
9,Group


In [4]:
def normalize_id(x):
    x = "" if pd.isna(x) else str(x)
    x = x.strip()
    x = re.sub(r"\s+", "", x)
    x = x.replace("[", "").replace("]", "")
    return x.upper()

candidate_terms = ["sample", "dog", "patient", "case", "cotc", "id", "specimen", "rna", "seq", "tumor"]
candidate_id_cols = [
    c for c in clinical_raw.columns
    if any(term in str(c).lower() for term in candidate_terms)
]

print("Candidate ID columns:")
display(pd.DataFrame({"candidate_id_column": candidate_id_cols}))

for c in candidate_id_cols:
    vals = clinical_raw[c].dropna().astype(str).head(10).tolist()
    print("\n", c)
    print(vals)

Candidate ID columns:


,candidate_id_column
0,Patient ID
1,Tumor Location



 Patient ID
['514', '518', '619', '639', '702', '710', '716', '719', '720', '721']

 Tumor Location
['Left distal radius', 'Left proximal humerus', 'Left distal tibia', 'Left distal femur', 'Right distal ulna', 'Left distal femur', 'Left distal radius', 'Right distal radius', 'Right distal femur', 'Right proximal tibia']


In [5]:
sample_map_path = PROCESSED_DIR / "GSE238110_sample_id_map.csv"
sample_map = pd.read_csv(sample_map_path)
sample_map["norm_bracket_id"] = sample_map["sample_id_bracket"].map(normalize_id)
sample_map["norm_original_column"] = sample_map["original_sample_column"].map(normalize_id)

overlap_results = []

for c in candidate_id_cols:
    values = clinical_raw[c].map(normalize_id)
    overlap_bracket = values.isin(set(sample_map["norm_bracket_id"])).sum()
    overlap_original = values.isin(set(sample_map["norm_original_column"])).sum()
    overlap_results.append({
        "clinical_column": c,
        "overlap_with_bracket_ids": int(overlap_bracket),
        "overlap_with_original_columns": int(overlap_original),
        "non_missing": int(clinical_raw[c].notna().sum())
    })

overlap_df = pd.DataFrame(overlap_results).sort_values(
    ["overlap_with_bracket_ids", "overlap_with_original_columns", "non_missing"],
    ascending=False
)

display(overlap_df)
overlap_df.to_csv(PROCESSED_DIR / "DOG2_clinical_id_overlap_candidates.csv", index=False)

,clinical_column,overlap_with_bracket_ids,overlap_with_original_columns,non_missing
0,Patient ID,0,0,186
1,Tumor Location,0,0,186


In [6]:
def normalize_patient_id(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    m = re.search(r"\d+", x)
    if m is None:
        return None
    return str(int(m.group(0)))


def extract_patient_id_from_expression_col(x):
    x = str(x).strip()
    m = re.match(r"^\d+_0*(\d+)", x)
    if m is None:
        return None
    return str(int(m.group(1)))


def classify_sample_type(x):
    x_low = str(x).lower()
    if "met" in x_low or "me_t" in x_low:
        return "metastasis_or_met_related"
    if "tumor" in x_low:
        return "primary_or_tumor"
    return "other"


sample_map = pd.read_csv(PROCESSED_DIR / "GSE238110_sample_id_map.csv")

sample_map["patient_id_clean"] = sample_map["original_sample_column"].map(
    extract_patient_id_from_expression_col
)
sample_map["sample_type"] = sample_map["original_sample_column"].map(classify_sample_type)

clinical = clinical_raw.copy()
clinical["patient_id_clean"] = clinical["Patient ID"].map(normalize_patient_id)

display(sample_map.head(20))
display(clinical[["Patient ID", "patient_id_clean", "DFS_time", "DFS_status", "OS_time", "OS_status"]].head())


,original_sample_column,sample_id_bracket,patient_id_clean,sample_type
0,14_0614__Tumor,14_0614__Tumor,614,primary_or_tumor
1,5_0220__Tumor,5_0220__Tumor,220,primary_or_tumor
2,13_0613__Tumor,13_0613__Tumor,613,primary_or_tumor
3,26_0722__Tumor,26_0722__Tumor,722,primary_or_tumor
4,4_0214__Tumor,4_0214__Tumor,214,primary_or_tumor
5,10_0512__Tumor,10_0512__Tumor,512,primary_or_tumor
6,32_0904__Tumor,32_0904__Tumor,904,primary_or_tumor
7,49_1803__Tumor,49_1803__Tumor,1803,primary_or_tumor
8,9_0510__Tumor,9_0510__Tumor,510,primary_or_tumor
9,18_0620_Me_T__Skin,18_0620_Me_T__Skin,620,metastasis_or_met_related


,Patient ID,patient_id_clean,DFS_time,DFS_status,OS_time,OS_status
0,514,514,33,0,117,1
1,518,518,12,0,248,0
2,619,619,63,1,622,0
3,639,639,234,1,283,1
4,702,702,153,1,171,0


In [7]:
sample_priority = {
    "primary_or_tumor": 0,
    "metastasis_or_met_related": 1,
    "other": 2,
}

sample_map["sample_priority"] = sample_map["sample_type"].map(sample_priority).fillna(9)

duplicates = (
    sample_map
    .dropna(subset=["patient_id_clean"])
    .groupby("patient_id_clean")
    .size()
    .sort_values(ascending=False)
)

print("Expression samples:", sample_map.shape[0])
print("Unique expression patient IDs:", sample_map["patient_id_clean"].nunique())
print("Patients with multiple expression samples:", int((duplicates > 1).sum()))

display(duplicates[duplicates > 1].head(20).to_frame("n_expression_samples"))


Expression samples: 198
Unique expression patient IDs: 195
Patients with multiple expression samples: 3


,n_expression_samples
patient_id_clean,
1022,2
620,2
621,2


In [8]:
sample_one_per_patient = (
    sample_map
    .dropna(subset=["patient_id_clean"])
    .sort_values(["patient_id_clean", "sample_priority", "original_sample_column"])
    .drop_duplicates("patient_id_clean", keep="first")
)

matched_clinical = clinical.merge(
    sample_one_per_patient[
        ["patient_id_clean", "original_sample_column", "sample_id_bracket", "sample_type"]
    ],
    on="patient_id_clean",
    how="inner"
)

print("Clinical rows:", clinical.shape[0])
print("Matched clinical-expression samples:", matched_clinical.shape[0])

unmatched_clinical = sorted(set(clinical["patient_id_clean"].dropna()) - set(sample_one_per_patient["patient_id_clean"].dropna()))
unmatched_expression = sorted(set(sample_one_per_patient["patient_id_clean"].dropna()) - set(clinical["patient_id_clean"].dropna()))

print("Unmatched clinical IDs:", len(unmatched_clinical))
print("Unmatched expression IDs:", len(unmatched_expression))

display(matched_clinical.head())
display(pd.DataFrame({"unmatched_clinical_id": unmatched_clinical[:30]}))
display(pd.DataFrame({"unmatched_expression_id": unmatched_expression[:30]}))


Clinical rows: 186
Matched clinical-expression samples: 186
Unmatched clinical IDs: 0
Unmatched expression IDs: 9


,Patient ID,Tumor Location,Site,age,weight,breed,gender,PH,ALP,Group,DFS_time,DFS_status,OS_time,OS_status,treatment,primary_immune_subtype,patient_id_clean,original_sample_column,sample_id_bracket,sample_type
0,514,Left distal radius,UW,5.5,42.7,Mixed Breed,Spayed Female,0,Normal,NPHNALP,33,0,117,1,SOC,ID,514,100_0514_tumor,100_0514_tumor,primary_or_tumor
1,518,Left proximal humerus,UW,8.3,44.3,Rottweiler,Spayed Female,1,Elevated,PHEALP,12,0,248,0,SOC,IE-ECM,518,101_0518_tumor,101_0518_tumor,primary_or_tumor
2,619,Left distal tibia,OSU,7.5,30.8,Mixed Breed,Spayed Female,0,Normal,NPHNALP,63,1,622,0,SOC,IE-ECM,619,102_0619_tumor,102_0619_tumor,primary_or_tumor
3,639,Left distal femur,OSU,6.9,36.2,Greyhound,Castrated Male,0,Normal,NPHNALP,234,1,283,1,SOC,ID,639,103_0639_tumor,103_0639_tumor,primary_or_tumor
4,702,Right distal ulna,UIL,9.5,33.2,Labrador Retriever,Castrated Male,0,Normal,NPHNALP,153,1,171,0,SOC,IE-ECM,702,104_0702_tumor,104_0702_tumor,primary_or_tumor


,unmatched_clinical_id


,unmatched_expression_id
0,1701
1,1702
2,1706
3,227
4,310
5,327
6,628
7,811
8,902


In [9]:
clinical_export = matched_clinical.copy()

clinical_export["dfi_time"] = pd.to_numeric(clinical_export["DFS_time"], errors="coerce")
clinical_export["dfi_event"] = pd.to_numeric(clinical_export["DFS_status"], errors="coerce")

clinical_export["os_time"] = pd.to_numeric(clinical_export["OS_time"], errors="coerce")
clinical_export["os_event"] = pd.to_numeric(clinical_export["OS_status"], errors="coerce")


def event_by_horizon(time, event, horizon_days):
    if pd.isna(time) or pd.isna(event):
        return np.nan
    if event == 1 and time <= horizon_days:
        return 1
    if time >= horizon_days:
        return 0
    return np.nan


clinical_export["dfi_event_365d"] = [
    event_by_horizon(t, e, 365)
    for t, e in zip(clinical_export["dfi_time"], clinical_export["dfi_event"])
]

clinical_export["dfi_event_540d"] = [
    event_by_horizon(t, e, 540)
    for t, e in zip(clinical_export["dfi_time"], clinical_export["dfi_event"])
]

endpoint_summary = pd.DataFrame({
    "endpoint": ["dfi_event", "os_event", "dfi_event_365d", "dfi_event_540d"],
    "non_missing": [
        clinical_export["dfi_event"].notna().sum(),
        clinical_export["os_event"].notna().sum(),
        clinical_export["dfi_event_365d"].notna().sum(),
        clinical_export["dfi_event_540d"].notna().sum(),
    ],
    "events_or_positive": [
        clinical_export["dfi_event"].sum(),
        clinical_export["os_event"].sum(),
        clinical_export["dfi_event_365d"].sum(),
        clinical_export["dfi_event_540d"].sum(),
    ],
})

display(endpoint_summary)

clinical_export.to_csv(
    PROCESSED_DIR / "DOG2_clinical_matched_to_GSE238110.csv",
    index=False
)

print("Saved:", PROCESSED_DIR / "DOG2_clinical_matched_to_GSE238110.csv")


,endpoint,non_missing,events_or_positive
0,dfi_event,186,143.0
1,os_event,186,124.0
2,dfi_event_365d,161,120.0
3,dfi_event_540d,161,136.0


Saved: C:\Users\olegk\Desktop\paper4_sarcoma_dog\data\processed\DOG2_clinical_matched_to_GSE238110.csv
